# tranSPORTmer_D · 짧은 데드볼과 인플레이 오탐 오버샘플링

- **비교** · 기존 데드볼 전용 모델, 짧은 데드볼 3배, 짧은 데드볼과 인플레이 오탐 3배.
- **입력** · 공모전 폴더의 공개 DFL 7경기 Parquet. 9.6초 선수 좌표 창.
- **평가** · 기존과 동일한 6경기 학습·1경기 평가, 조기 종료, 25fps 평가와 비터비·라인아웃 보정.
- **실행** · 기존 데드볼 전용 실험과 같은 Python 환경에서 셀을 순서대로 실행한다.

In [ ]:
import json
from pathlib import Path

root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/train_dfl.py").is_file())
state_notebook = root / "model/영상추출/모델/dead_ball/tranSPORTmer_D.ipynb"
state_cells = {cell["id"]: cell for cell in json.loads(state_notebook.read_text())["cells"]}
for index in (2, 4):
    source = "".join(state_cells[f"transportmer-d-{index}"]["source"])
    exec(compile(source, f"{state_notebook}:{index}", "exec"))

BASELINE_OUT = OUT
BASELINE = json.loads((BASELINE_OUT / "run_manifest.json").read_text())
assert digest(state_notebook) == BASELINE["notebook_sha256"]
assert config == {key: BASELINE["config"][key] for key in config}
RUN_ID = "transportmer_d_oversampling_20261112_01"
RESULTS = BASELINE_OUT.parent / RUN_ID
NOTEBOOK = folder / "tranSPORTmer_D_oversampling.ipynb"
WEIGHT = 3.0
SHORT_MAX_S = 10.0
BOUNDARY_RADIUS_S = 1.0
MIN_FALSE_DEAD_S = 2.0
MINING_EPOCH = 5
VARIANTS = ("short", "short_and_false_dead")

## 1. 학습 창 선별

- **짧은 데드볼** · 10초 미만 중단의 시작이나 재개 ±1초에 닿는 학습 창. 중단과 양쪽 문맥이 학습 구간 안에 있을 때만 사용한다.
- **인플레이 오탐** · C는 첫 5 epoch를 B와 같은 방식으로 학습한다. 그 모델이 학습용 인플레이 창에서 연속 2초 이상 데드볼 확률 0.5 이상을 낸 창을 선별한다.
- **3배 가중치** · 에폭마다 기존과 같은 수의 창을 뽑는다. 어려운 창을 한 번씩 포함하고, 일부 일반 창을 어려운 창으로 교체해 두 그룹의 창당 기대 추출 횟수 비율을 3:1로 맞춘다.

In [ ]:
def short_windows(data, train_indices, spells):
    rows = data["indices"][train_indices]
    valid = data["valid"][train_indices]
    starts = rows[:, 0]
    period = data["period"]
    half_ends = {period[half.start]: half.stop for half in halves(period)}
    stops = np.minimum(
        rows[np.arange(len(rows)), valid.sum(axis=1) - 1] + STEP,
        [half_ends[p] for p in period[starts]],
    )
    split = (starts[1:] != stops[:-1]) | (period[starts[1:]] != period[starts[:-1]])
    cuts = np.r_[0, np.flatnonzero(split) + 1, len(rows)]
    selected = np.zeros(len(rows), dtype=bool)
    radius = round(BOUNDARY_RADIUS_S * FPS)
    short = spells[spells.seconds < SHORT_MAX_S]
    for first, last in zip(cuts[:-1], cuts[1:]):
        left, right = starts[first:last], stops[first:last]
        eligible = short[(short.start - radius >= left[0]) & (short.stop + radius <= right[-1])]
        for spell in eligible.itertuples(index=False):
            for boundary in (spell.start, spell.stop):
                selected[first:last] |= (left < boundary + radius) & (right > boundary - radius)
    return selected


def false_dead_windows(model, train):
    live = np.flatnonzero(train["valid"].all(axis=1) & (train["target"] == 0).all(axis=1))
    selected = np.zeros(len(train["target"]), dtype=bool)
    if not len(live):
        return selected
    probability, _ = model_predictions(model, train, live)
    minimum = int(np.ceil(MIN_FALSE_DEAD_S * FPS / STEP))
    for index, dead in zip(live, probability >= 0.5):
        selected[index] = any(stop - start >= minimum for start, stop in features.runs_of(dead))
    return selected


def weighted_order(selected, rng):
    hard = np.flatnonzero(selected)
    ordinary = np.flatnonzero(~selected)
    size = len(selected)
    if not len(hard) or not len(ordinary):
        return rng.permutation(size)
    hard_count = size * WEIGHT * len(hard) / (WEIGHT * len(hard) + len(ordinary))
    extra = int(hard_count - len(hard) + rng.random())
    order = np.concatenate((
        hard, rng.choice(hard, extra, replace=True),
        rng.choice(ordinary, len(ordinary) - extra, replace=False),
    ))
    return rng.permutation(order)


def check_sampler():
    selected = np.arange(1000) < 80
    rng = np.random.default_rng(SEED)
    draws = []
    for _ in range(100):
        order = weighted_order(selected, rng)
        assert len(order) == len(selected)
        assert len(np.unique(order[~selected[order]])) == (~selected[order]).sum()
        assert np.bincount(order, minlength=len(selected))[selected].min() >= 1
        draws.append(selected[order].sum())
    hard_mean = float(np.mean(draws))
    ratio = (hard_mean / selected.sum()) / ((len(selected) - hard_mean) / (~selected).sum())
    assert abs(ratio - WEIGHT) < 0.01, ratio

## 2. 데드볼 학습과 평가

데드볼 BCE, 모델과 최적화 설정을 재사용한다. 오탐 창은 학습에 쓴 창에서만 찾는다. 세 조건에서 인플레이 Spell F1, 데드볼 프레임 F1, 짧은 중단 검출과 인플레이 오탐을 비교한다.

In [ ]:
def fit_weighted(held, variant, matches, datasets, splits, short_masks):
    model, opt = new_joint()
    train = {key: np.concatenate([datasets[m][key][splits[m][0]] for m in matches])
             for key in ("x", "valid", "target")}
    short = np.concatenate([short_masks[m] for m in matches])
    false = np.zeros_like(short)
    sampling = dict(windows=len(short), short_windows=int(short.sum()), false_windows=0,
                    samples=0, short_draws=0, false_draws=0, mining_s=0.0)
    assert 0 < short.sum() < len(short)
    best_score, best_epoch, best, stale = -np.inf, 0, None, 0
    started = perf_counter()

    for epoch in range(1, EPOCHS + 1):
        tick = perf_counter()
        if variant == "short_and_false_dead" and epoch == MINING_EPOCH + 1:
            false = false_dead_windows(model, train)
            sampling.update(false_windows=int(false.sum()), mining_s=perf_counter() - tick)
            print(f"  {held} 인플레이 오탐 창 {false.sum():,}/{len(false):,}", flush=True)
        model.train()
        order = weighted_order(short | false, np.random.default_rng(SEED + epoch))
        sampling["samples"] += len(order)
        sampling["short_draws"] += int(short[order].sum())
        sampling["false_draws"] += int(false[order].sum())
        losses = [train_state_batch(model, opt, train, order[i:i+BATCH])
                  for i in range(0, len(order), BATCH)]
        bce = float(torch.stack(losses).mean().item())
        assert np.isfinite(bce)
        validation = inner_f1(model, datasets, splits, matches) if epoch >= MIN_EPOCHS else None
        if validation is not None:
            if validation > best_score + MIN_DELTA:
                best_score, best_epoch, best, stale = validation, epoch, cpu_weights(model), 0
            else:
                stale += 1
        state = "대기" if validation is None else f"{validation:.4f}"
        print(f"  {variant} {held} epoch {epoch:02}/{EPOCHS} | {perf_counter()-tick:.1f}s | "
              f"BCE {bce:.4f} | 내부 Spell F1 {state} | best {best_epoch} | patience {stale}/{PATIENCE}",
              flush=True)
        if stale >= PATIENCE:
            break

    assert best is not None and best_epoch >= MIN_EPOCHS
    assert sampling["samples"] == epoch * len(short)
    model.load_state_dict(best)
    return model.eval(), epoch, best_epoch, best_score, perf_counter() - started, sampling


def diagnostics(evaluation, state):
    spans = evaluation["spells"]
    found = np.array([state[span.start:span.stop].mean() >= 0.5 for span in spans.itertuples()])
    label = evaluation["label"]
    row = dict(false_dead=sum(
        int(label[half][start:stop].mean() < 0.5)
        for half in halves(evaluation["period"])
        for start, stop in features.runs_of(state[half] == 1)
    ))
    seconds = spans.seconds.to_numpy()
    for name, mask in (
        ("under_3s", seconds < 3.0),
        ("3_to_10s", (seconds >= 3.0) & (seconds < SHORT_MAX_S)),
        ("under_10s", seconds < SHORT_MAX_S),
    ):
        row[f"{name}_found"] = int((found & mask).sum())
        row[f"{name}_total"] = int(mask.sum())
    return row

## 3. 실행과 결과

경기별 결과를 확정할 때마다 집계표와 manifest를 갱신한다. 프레임 CSV는 각 조건의 `matches/<경기ID>/`에 저장한다. 이미 확정된 결과는 해시가 일치하면 재실행에서 재사용한다.

In [ ]:
def prepare_data():
    datasets, evaluations, splits, masks, inputs = {}, {}, {}, {}, {}
    for match in MATCHES:
        for path in map(Path, dfl_load.cache_paths(match)):
            inputs[str(path.relative_to(ROOT))] = digest(path)
        datasets[match], evaluations[match] = prepare_joint(match)
        splits[match] = internal_split(datasets[match])
        for key in ("ball", "ball_target"):
            datasets[match].pop(key)
        masks[match] = short_windows(datasets[match], splits[match][0], evaluations[match]["spells"])
        print(f"{match} | 짧은 중단 대상 창 {masks[match].sum():,}/{len(masks[match]):,}", flush=True)
    assert inputs == BASELINE["input_sha256"]
    assert all(0 < mask.sum() < len(mask) for mask in masks.values())
    return datasets, evaluations, splits, masks, inputs


def detail_table(folds):
    return pd.DataFrame([
        dict(match=fold["match"], best_epoch=fold["best_epoch"], stopped_epoch=fold["stopped_epoch"],
             training_s=fold["training_s"], fold_s=fold["fold_s"], **score)
        for fold in folds for score in fold["scores"]
    ])


def score_table(detail, variant):
    counts = ["false_dead", *[f"{name}_{value}" for name in ("under_3s", "3_to_10s", "under_10s")
                            for value in ("found", "total")]]
    summary = (detail.groupby("stage", sort=False)
               .agg(dead_frame_f1=("dead_f1", "mean"), alive_spell_f1=("alive_spell_f1", "mean"),
                    training_s=("training_s", "sum"), fold_s=("fold_s", "sum"),
                    **{key: (key, "sum") for key in counts}).reset_index())
    for name in ("under_3s", "3_to_10s", "under_10s"):
        summary[f"{name}_recall"] = summary[f"{name}_found"] / summary[f"{name}_total"]
    return summary.assign(variant=variant)


def write_table(table, path):
    pending = path.with_suffix(".tmp")
    table.to_csv(pending, index=False)
    pending.replace(path)


def baseline_scores(evaluations):
    rows = []
    for match in MATCHES:
        probability = pd.read_csv(BASELINE_OUT / "matches" / match / "features.csv")
        np.testing.assert_array_equal(probability.source_frame_id, evaluations[match]["source_frame"])
        scores, viterbi, forced = score_joint(evaluations[match], probability.dead_score.to_numpy())
        np.testing.assert_array_equal(viterbi, probability.dead_viterbi)
        np.testing.assert_array_equal(forced, probability.dead)
        for row, state in zip(scores, (probability.dead_score.to_numpy() >= 0.5, viterbi, forced)):
            row.update(diagnostics(evaluations[match], state))
        rows.append(dict(next(fold for fold in BASELINE["matches"] if fold["match"] == match), scores=scores))
    detail = detail_table(rows)
    summary = score_table(detail, "baseline")
    expected = pd.read_csv(BASELINE_OUT / "model_summary.csv")
    np.testing.assert_allclose(summary[["dead_frame_f1", "alive_spell_f1"]],
                               expected[["dead_frame_f1", "alive_spell_f1"]], atol=1e-12)
    return summary


def run_variant(variant, fingerprint, datasets, evaluations, splits, masks):
    global OUT
    OUT = RESULTS / variant
    OUT.mkdir(parents=True, exist_ok=True)
    manifest_path = OUT / "run_manifest.json"
    manifest = dict(run_id=RUN_ID, variant=variant, fingerprint=fingerprint, matches=[])
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text())
        assert manifest["fingerprint"] == fingerprint and manifest["variant"] == variant
    folds = manifest["matches"]
    assert len({fold["match"] for fold in folds}) == len(folds)
    completed = {fold["match"] for fold in folds}
    for fold in folds:
        assert all(digest(OUT / path) == sha for path, sha in fold["file_sha256"].items())

    for held in MATCHES:
        if held in completed:
            continue
        started = perf_counter()
        matches = [match for match in MATCHES if match != held]
        assert len(matches) == 6
        print(f"{variant} | {held} 바깥 평가 | 학습/내부 검증 {', '.join(matches)}", flush=True)
        model, stopped, best, inner, fit_s, sampling = fit_weighted(
            held, variant, matches, datasets, splits, masks,
        )
        evaluation = evaluations[held]
        probability, sampled = predict(StateOutput(model), evaluation)
        scores, viterbi, forced = score_joint(evaluation, probability)
        for row, state in zip(scores, (probability >= 0.5, viterbi, forced)):
            row.update(diagnostics(evaluation, state))
        files = write_frames(held, evaluation, probability, sampled, viterbi, forced)
        fold = dict(match=held, train_matches=matches, best_epoch=best, stopped_epoch=stopped,
                    best_inner_f1=inner, frames=len(probability), training_s=fit_s,
                    fold_s=perf_counter()-started, sampling=sampling, scores=scores,
                    file_sha256={str(path.relative_to(OUT)): digest(path) for path in files})
        folds.append(fold)
        save_json(manifest, manifest_path)
        write_table(detail_table(folds), OUT / "per_match.csv")
        final = scores[-1]
        print(f"{variant} | {held} 완료 | epoch {best}/{stopped} | "
              f"프레임 F1 {final['dead_f1']:.4f} | Spell F1 {final['alive_spell_f1']:.4f} | "
              f"10초 미만 {final['under_10s_found']}/{final['under_10s_total']} | "
              f"인플레이 오탐 {final['false_dead']} | {fold['fold_s']:.1f}s", flush=True)
        del model

    detail = detail_table(folds)
    assert set(detail.match) == set(MATCHES)
    assert len(detail) == 21 and not detail.duplicated(["match", "stage"]).any()
    assert detail.dead_f1.between(0, 1).all() and detail.alive_spell_f1.between(0, 1).all()
    summary = score_table(detail, variant)
    write_table(detail, OUT / "per_match.csv")
    write_table(summary, OUT / "model_summary.csv")
    names = ("per_match.csv", "model_summary.csv")
    manifest["report_sha256"] = {name: digest(OUT / name) for name in names}
    save_json(manifest, manifest_path)
    save_json(dict(passed=True, held_matches=len(folds), score_rows=len(detail),
                   frames=sum(fold["frames"] for fold in folds), smoke_test=True), OUT / "validation.json")
    return summary


def run_oversampling(parameters):
    datasets, evaluations, splits, masks, inputs = prepare_data()
    fingerprint = dict(notebook_sha256=digest(NOTEBOOK), state_notebook_sha256=digest(state_notebook),
                       baseline_manifest_sha256=digest(BASELINE_OUT / "run_manifest.json"),
                       joint_notebook_sha256=digest(joint), base_notebook_sha256=digest(original),
                       input_sha256=inputs, device=str(DEVICE), amp=AMP, config=dict(
                           config, ball_loss_weight=0, model_parameters=parameters, weight=WEIGHT,
                           short_max_s=SHORT_MAX_S, boundary_radius_s=BOUNDARY_RADIUS_S,
                           min_false_dead_s=MIN_FALSE_DEAD_S, mining_epoch=MINING_EPOCH,
                       ), versions={name: version(name) for name in ("torch", "pandas", "numpy", "pyarrow")},
                       code_commit=subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip())
    comparison = baseline_scores(evaluations)
    for variant in VARIANTS:
        summary = run_variant(variant, fingerprint, datasets, evaluations, splits, masks)
        comparison = pd.concat([comparison, summary], ignore_index=True)
        write_table(comparison, RESULTS / "comparison.csv")
    print(comparison.query("stage == 'line_out'").round(4).to_string(index=False), flush=True)
    return comparison


check_sampler()
parameters = verify_state_only()
print(f"device={DEVICE} | params={parameters:,} | sampling weight {WEIGHT:g} | ball loss 0", flush=True)
if os.environ.get("MTAH_TRANSPORTMER_OVERSAMPLING_MODE") == "check":
    _, evaluations, _, _, _ = prepare_data()
    print(baseline_scores(evaluations).query("stage == 'line_out'").round(4).to_string(index=False), flush=True)
else:
    comparison = run_oversampling(parameters)